[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/058.%20The%20ABC_XYZ%20Inventory%20Classification%20Problem/P58-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 58. The ABC/XYZ Inventory Classification Problem

## Tier 2: A Probabilistic Heuristic in Python

### Purpose and workflow

This notebook classifies stock keeping units (SKUs) along two dimensions. The ABC dimension reflects each SKU's annual consumption value relative to the portfolio; the XYZ dimension reflects demand variability, measured by the coefficient of variation (CV). Gaussian transition functions turn threshold-based classifications into probability distributions, and weighted random selection produces one ABC class and one XYZ class for each SKU.

The workflow is:

1. Compute portfolio-based ABC value thresholds using cumulative annual value.
2. Convert distance from ABC and XYZ thresholds into class probabilities.
3. Independently sample an ABC class and an XYZ class using those probabilities.
4. Report the nine combined category probabilities, a heuristic confidence score, and a boundary-case flag.
5. Examine results, visualize the probability transitions, and assess the repeatability of sampled classifications.

### Scope and modeling assumptions

- Annual values are nonnegative and represent comparable annual consumption values. ABC thresholds are the annual values of the SKUs at which cumulative portfolio value first reaches the specified proportions.
- XYZ classification uses the sample standard deviation of the supplied monthly demands and divides it by their arithmetic mean. The monthly observations are treated as a descriptive history; no time-series or demand-generation model is fitted.
- Gaussian transitions are a heuristic mapping from a measured value to class weights. They are not estimated from historical classification errors and do not, by themselves, establish calibrated probabilities about future outcomes.
- The ABC and XYZ class draws are independent conditional on their respective probability vectors. Accordingly, a combined probability is computed as the product of the two marginal probabilities.
- The confidence score is a summary index derived from the class-probability distributions. It is not a statistical guarantee, a calibrated probability of correctness, or a forecast interval.
- The method does not model ordering costs, lead times, service levels, stockout penalties, or inventory-control decisions. It classifies SKUs; it does not optimize inventory policy.

### Background: ABC, XYZ, and probability distributions

ABC analysis groups items by their contribution to a chosen value measure. In this notebook, SKUs are sorted from greatest to least annual value, and cumulative value shares determine the A and B cutoffs; the remaining items fall toward C. XYZ analysis groups items by demand variability: a lower CV indicates more stable demand, while a higher CV indicates more variable demand.

A probability distribution over classes assigns a nonnegative weight to each possible class, with all weights summing to one. Here, the weights express the heuristic's degree of support for each class near a threshold. Random selection uses these weights, so a class with twice the weight is twice as likely to be drawn in that classification step.

### Mathematical formulation

Let \(\mathcal{S}\) be the set of SKUs. For SKU \(i\in\mathcal{S}\), let \(v_i\) be annual value and let \(d_{it}\) be demand in month \(t\), for \(t=1,\ldots,T\). The sample mean and sample standard deviation of monthly demand are

$$
\bar d_i=\frac{1}{T}\sum_{t=1}^{T}d_{it},\qquad
s_i=\sqrt{\frac{1}{T-1}\sum_{t=1}^{T}(d_{it}-\bar d_i)^2},\qquad
CV_i=\begin{cases}s_i/\bar d_i,&\bar d_i>0,\\+\infty,&\bar d_i=0.\end{cases}
$$

The code computes these statistics in `SKU.__post_init__`, using `numpy.std(..., ddof=1)` for the sample standard deviation.

Sort the SKUs in descending order of annual value. If \(k_\alpha\) is the first position whose cumulative share of total annual value is at least \(\alpha\), and \(k_\beta\) is the corresponding position for \(\beta\), the value thresholds are

$$
a= v_{(k_\alpha)},\qquad b=v_{(k_\beta)},\qquad 0<\alpha<\beta<1.
$$

Here \(a\) and \(b\) are the annual values at the cumulative-share cutoffs. They are not themselves the cumulative shares. The implementation obtains them in `probabilistic_abc_xyz_classification` using `abc_alpha` and `abc_beta`.

For a threshold \(q\), transition half-width \(w>0\), and a measured value \(x\), define the Gaussian transition weight

$$
G_{\uparrow}(x;q,w)=\Phi\!\left(\frac{x-q}{w/2}\right),\qquad
G_{\downarrow}(x;q,w)=\Phi\!\left(\frac{q-x}{w/2}\right),
$$

where \(\Phi\) is the standard normal cumulative distribution function. The upward function supports values above a threshold; the downward function supports values below it. In the code, `gaussian_transition_function` evaluates these functions. The scale \(w/2\) makes the standard deviation of the transition curve half the specified half-width; the curve is smooth and approaches, but does not abruptly clip to, zero or one.

For ABC, the code sets \(w_{ABC}=\texttt{transition\_width}\max(a,b)\) and forms the initial weights

$$
r_A=G_{\uparrow}(v_i;a,w_{ABC}),\qquad
r_C=G_{\downarrow}(v_i;b,w_{ABC}),\qquad
r_B=\max(0,1-r_A-r_C).
$$

For XYZ, let \(\gamma_1\) and \(\gamma_2\) be the CV thresholds and let \(w_{XYZ}=0.02\). The initial weights are

$$
r_X=G_{\downarrow}(CV_i;\gamma_1,w_{XYZ}),\qquad
r_Z=G_{\uparrow}(CV_i;\gamma_2,w_{XYZ}),\qquad
r_Y=\max(0,1-r_X-r_Z).
$$

In each dimension, the three initial weights are normalized by their sum to obtain probabilities. This residual-and-normalization rule keeps the probability vector valid even if the transition regions overlap. The functions `calculate_abc_probabilities` and `calculate_xyz_probabilities` implement these calculations.

Given the normalized marginal probabilities \(p_i(A),p_i(B),p_i(C)\) and \(p_i(X),p_i(Y),p_i(Z)\), the method assumes conditional independence between the two classification draws:

$$
p_i(uv)=p_i(u)p_i(v),\qquad u\in\{A,B,C\},\quad v\in\{X,Y,Z\}.
$$

The nine joint weights sum to one. The function `calculate_combined_probabilities` calculates them, while `randomized_classification_selection` samples each dimension separately. The sampled pair is the reported category; it need not be the most probable combined category.

The confidence score combines the average of the two largest marginal class probabilities with a normalized entropy adjustment. For a three-class probability vector \(p\), its Shannon entropy is \(H(p)=-\sum_j p_j\log_2p_j\), with zero-probability terms treated as zero. The code uses

$$
C_i=\frac{m_{ABC}+m_{XYZ}}{2}\left[0.7+0.3\left(1-\frac{H(p_{ABC})+H(p_{XYZ})}{2\log_2 3}\right)\right],
$$

where \(m_{ABC}\) and \(m_{XYZ}\) are the maximum probabilities in their respective dimensions. This bounded score is a heuristic summary, not a calibrated probability. The function `calculate_confidence_score` implements it. A SKU is flagged as a boundary case if either marginal maximum is below the default threshold of 0.7, as implemented in `is_boundary_case`.

### Computational notes

For \(n\) SKUs, sorting for the ABC thresholds takes \(O(n\log n)\) time. Once thresholds are available, probability calculation and sampling take \(O(n)\) time, apart from constant-size operations over the three classes. The method stores a constant number of probabilities per SKU, so its result storage is \(O(n)\). Randomized assignments can differ between runs; a fixed random seed makes a particular execution reproducible, while repeated runs reveal how assignments vary under the specified probability weights.

### Interpreting the analysis

The transition plots show how class weights change with measured value or CV. A diffuse probability distribution indicates that the heuristic gives substantial weight to multiple classes; it does not prove that the underlying data are uncertain in a statistical sense. The stability analysis compares sampled labels with one reference run for a subset of SKUs. Its consistency measure is an observed fraction of matching labels, not a guarantee of future stability.

The examples and dataset are generated within the notebook. Their results illustrate the specified heuristic and should not be interpreted as universal ABC/XYZ thresholds or as evidence of business performance.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.stats import norm

# A fixed seed makes the generated example and its initial randomized labels reproducible.
np.random.seed(42)

plt.style.use("default")
sns.set_palette("husl")

print("Libraries imported successfully for probabilistic ABC/XYZ classification.")

In [ ]:
@dataclass
class SKU:
    """A stock keeping unit with annual value and monthly demand observations."""

    sku_id: str
    annual_value: float
    monthly_demands: List[float]

    def __post_init__(self) -> None:
        """Validate the observations and calculate descriptive demand statistics."""
        if not self.sku_id:
            raise ValueError("sku_id must be a non-empty string.")
        if not np.isfinite(self.annual_value) or self.annual_value < 0:
            raise ValueError("annual_value must be finite and nonnegative.")
        if len(self.monthly_demands) < 2:
            raise ValueError("At least two monthly demand observations are required.")

        demands = np.asarray(self.monthly_demands, dtype=float)
        if not np.all(np.isfinite(demands)) or np.any(demands < 0):
            raise ValueError("Monthly demands must be finite and nonnegative.")

        self.avg_demand = float(np.mean(demands))
        self.std_demand = float(np.std(demands, ddof=1))
        self.cv = (
            self.std_demand / self.avg_demand
            if self.avg_demand > 0
            else float("inf")
        )


@dataclass
class ProbabilisticThresholds:
    """Cutoffs and transition width used by the probabilistic classification."""

    abc_alpha: float = 0.80
    abc_beta: float = 0.95
    xyz_gamma1: float = 0.10
    xyz_gamma2: float = 0.25
    transition_width: float = 0.05

    def __post_init__(self) -> None:
        """Ensure that the thresholds define ordered, meaningful boundaries."""
        if not 0 < self.abc_alpha < self.abc_beta < 1:
            raise ValueError("ABC cutoffs must satisfy 0 < abc_alpha < abc_beta < 1.")
        if not 0 <= self.xyz_gamma1 < self.xyz_gamma2:
            raise ValueError("XYZ CV thresholds must satisfy 0 <= gamma1 < gamma2.")
        if not np.isfinite(self.transition_width) or self.transition_width <= 0:
            raise ValueError("transition_width must be finite and positive.")


@dataclass
class ProbabilisticResult:
    """Probability summaries and one sampled ABC/XYZ label for a SKU."""

    sku: SKU
    abc_probabilities: Dict[str, float]
    xyz_probabilities: Dict[str, float]
    combined_probabilities: Dict[str, float]
    final_classification: str
    confidence_score: float
    is_boundary_case: bool

print("Data classes defined for probabilistic classification.")

In [ ]:
def gaussian_transition_function(
    value: float,
    threshold: float,
    width: float,
    direction: str = "above",
) -> float:
    """Return a Gaussian-CDF weight for one side of a classification threshold.

    ``width`` is the transition half-width; the Gaussian scale is half that
    value. Using the CDF over its full domain gives a continuous transition.
    """
    if not np.isfinite(width) or width <= 0:
        raise ValueError("width must be finite and positive.")
    if direction == "above":
        return float(norm.cdf((value - threshold) / (width / 2)))
    if direction == "below":
        return float(norm.cdf((threshold - value) / (width / 2)))
    raise ValueError("direction must be either 'above' or 'below'.")


def calculate_abc_probabilities(
    sku: SKU,
    a_threshold: float,
    b_threshold: float,
    thresholds: ProbabilisticThresholds,
) -> Dict[str, float]:
    """Calculate normalized A, B, and C weights for one SKU."""
    width = thresholds.transition_width * max(a_threshold, b_threshold)
    p_a = gaussian_transition_function(
        sku.annual_value, a_threshold, width, "above"
    )
    p_c = gaussian_transition_function(
        sku.annual_value, b_threshold, width, "below"
    )

    # B receives the remaining mass; normalization handles overlapping tails.
    p_b = max(0.0, 1.0 - p_a - p_c)
    total = p_a + p_b + p_c
    if total <= 0 or not np.isfinite(total):
        raise ValueError("ABC class weights must have a finite positive sum.")

    return {"A": p_a / total, "B": p_b / total, "C": p_c / total}


def calculate_xyz_probabilities(
    sku: SKU,
    thresholds: ProbabilisticThresholds,
) -> Dict[str, float]:
    """Calculate normalized X, Y, and Z weights from the SKU's demand CV."""
    cv_width = 0.02
    p_x = gaussian_transition_function(
        sku.cv, thresholds.xyz_gamma1, cv_width, "below"
    )
    p_z = gaussian_transition_function(
        sku.cv, thresholds.xyz_gamma2, cv_width, "above"
    )

    # Y receives the remaining mass; normalization also covers overlapping tails.
    p_y = max(0.0, 1.0 - p_x - p_z)
    total = p_x + p_y + p_z
    if total <= 0 or not np.isfinite(total):
        raise ValueError("XYZ class weights must have a finite positive sum.")

    return {"X": p_x / total, "Y": p_y / total, "Z": p_z / total}

print("Probability calculation functions defined.")

In [ ]:
def randomized_classification_selection(
    abc_probs: Dict[str, float],
    xyz_probs: Dict[str, float],
) -> Tuple[str, str]:
    """Sample one ABC class and one XYZ class from their probability vectors."""
    abc_classes = list(abc_probs.keys())
    abc_weights = list(abc_probs.values())
    xyz_classes = list(xyz_probs.keys())
    xyz_weights = list(xyz_probs.values())

    selected_abc = str(np.random.choice(abc_classes, p=abc_weights))
    selected_xyz = str(np.random.choice(xyz_classes, p=xyz_weights))
    return selected_abc, selected_xyz


def calculate_combined_probabilities(
    abc_probs: Dict[str, float],
    xyz_probs: Dict[str, float],
) -> Dict[str, float]:
    """Calculate nine joint class weights under conditional independence."""
    combined_probs = {}
    for abc_class, abc_prob in abc_probs.items():
        for xyz_class, xyz_prob in xyz_probs.items():
            combined_probs[abc_class + xyz_class] = abc_prob * xyz_prob
    return combined_probs


def calculate_confidence_score(
    abc_probs: Dict[str, float],
    xyz_probs: Dict[str, float],
) -> float:
    """Return the notebook's bounded, entropy-adjusted confidence index."""
    max_abc_prob = max(abc_probs.values())
    max_xyz_prob = max(xyz_probs.values())
    base_confidence = (max_abc_prob + max_xyz_prob) / 2

    abc_entropy = -sum(
        p * np.log2(p) if p > 0 else 0.0 for p in abc_probs.values()
    )
    xyz_entropy = -sum(
        p * np.log2(p) if p > 0 else 0.0 for p in xyz_probs.values()
    )
    max_entropy = np.log2(3)
    entropy_factor = 1.0 - (abc_entropy + xyz_entropy) / (2 * max_entropy)
    confidence = base_confidence * (0.7 + 0.3 * entropy_factor)
    return float(np.clip(confidence, 0.0, 1.0))


def is_boundary_case(
    abc_probs: Dict[str, float],
    xyz_probs: Dict[str, float],
    threshold: float = 0.7,
) -> bool:
    """Flag a SKU when either dimension has no class probability at the threshold."""
    if not 0 <= threshold <= 1:
        raise ValueError("threshold must lie between 0 and 1.")
    return max(abc_probs.values()) < threshold or max(xyz_probs.values()) < threshold

print("Sampling, joint-probability, and confidence functions defined.")

In [ ]:
def probabilistic_abc_xyz_classification(
    skus: List[SKU],
    thresholds: ProbabilisticThresholds,
) -> List[ProbabilisticResult]:
    """Classify SKUs using cumulative-value thresholds and randomized labels."""
    if not skus:
        raise ValueError("At least one SKU is required for classification.")

    total_value = sum(sku.annual_value for sku in skus)
    if total_value <= 0 or not np.isfinite(total_value):
        raise ValueError("Total annual value must be finite and positive.")

    # The value cutoffs are the SKU values where cumulative portfolio shares
    # first reach the configured ABC proportions.
    sorted_skus = sorted(skus, key=lambda sku: sku.annual_value, reverse=True)
    cumulative_values = []
    running_total = 0.0
    for sku in sorted_skus:
        running_total += sku.annual_value
        cumulative_values.append(running_total / total_value)

    a_threshold_idx = next(
        i for i, share in enumerate(cumulative_values)
        if share >= thresholds.abc_alpha
    )
    b_threshold_idx = next(
        i for i, share in enumerate(cumulative_values)
        if share >= thresholds.abc_beta
    )
    a_threshold = sorted_skus[a_threshold_idx].annual_value
    b_threshold = sorted_skus[b_threshold_idx].annual_value

    results = []
    for sku in skus:
        abc_probs = calculate_abc_probabilities(
            sku, a_threshold, b_threshold, thresholds
        )
        xyz_probs = calculate_xyz_probabilities(sku, thresholds)
        combined_probs = calculate_combined_probabilities(abc_probs, xyz_probs)

        selected_abc, selected_xyz = randomized_classification_selection(
            abc_probs, xyz_probs
        )
        confidence = calculate_confidence_score(abc_probs, xyz_probs)
        boundary = is_boundary_case(abc_probs, xyz_probs)

        result = ProbabilisticResult(
            sku=sku,
            abc_probabilities=abc_probs,
            xyz_probabilities=xyz_probs,
            combined_probabilities=combined_probs,
            final_classification=selected_abc + selected_xyz,
            confidence_score=confidence,
            is_boundary_case=boundary,
        )
        results.append(result)

        # These checks ensure the probability summaries remain coherent.
        assert np.isclose(sum(abc_probs.values()), 1.0)
        assert np.isclose(sum(xyz_probs.values()), 1.0)
        assert np.isclose(sum(combined_probs.values()), 1.0)

    return results

print("Probabilistic classification procedure defined.")

In [ ]:
# The fixed seed makes the synthetic demand and value example reproducible.
np.random.seed(42)

# Three illustrative SKUs and seven deliberately varied examples make the
# behavior near value and variability boundaries easier to inspect.
test_skus = [
    SKU("PREMIUM_LAPTOP", 450000, [85, 82, 88, 84, 86, 85, 87, 83, 85, 86, 84, 85]),
    SKU("OFFICE_CHAIR", 45000, [25, 22, 35, 28, 31, 26, 29, 24, 33, 27, 30, 26]),
    SKU("SEASONAL_DECOR", 5000, [2, 1, 0, 5, 8, 15, 45, 62, 35, 12, 3, 1]),
]

boundary_skus = [
    SKU("BUDGET_TABLET", 95000, [45, 43, 47, 44, 46, 45, 48, 44, 46, 45, 43, 44]),
    SKU("MID_RANGE_PHONE", 98000, [42, 48, 35, 51, 38, 45, 41, 49, 36, 44, 47, 40]),
    SKU("BASIC_MOUSE", 18000, [15, 18, 12, 16, 14, 17, 13, 15, 16, 14, 18, 12]),
    SKU("CHEAP_KEYBOARD", 22000, [8, 12, 6, 10, 9, 11, 7, 9, 10, 8, 11, 9]),
    SKU("STABLE_MONITOR", 75000, [22, 21, 23, 22, 21, 22, 23, 21, 22, 23, 22, 21]),
    SKU("MODERATE_SPEAKER", 35000, [18, 22, 15, 25, 20, 17, 23, 19, 21, 16, 24, 18]),
    SKU("VOLATILE_GADGET", 15000, [3, 8, 1, 12, 15, 4, 9, 2, 11, 6, 14, 5]),
]
test_skus.extend(boundary_skus)

# Add synthetic SKUs across value bands to provide a broader demonstration set.
for i in range(90):
    value_range = np.random.choice(["high", "medium", "low"], p=[0.2, 0.3, 0.5])

    if value_range == "high":
        annual_value = np.random.uniform(80000, 500000)
        base_demand = np.random.uniform(50, 150)
        cv_target = np.random.uniform(0.02, 0.15)
    elif value_range == "medium":
        annual_value = np.random.uniform(20000, 80000)
        base_demand = np.random.uniform(20, 80)
        cv_target = np.random.uniform(0.08, 0.30)
    else:
        annual_value = np.random.uniform(5000, 25000)
        base_demand = np.random.uniform(5, 30)
        cv_target = np.random.uniform(0.15, 0.80)

    std_demand = base_demand * cv_target
    monthly_demands = np.random.normal(
        base_demand, std_demand, 12
    ).clip(0, None).tolist()
    test_skus.append(SKU(f"SKU_{i + 100:03d}", annual_value, monthly_demands))

prob_thresholds = ProbabilisticThresholds(
    abc_alpha=0.80,
    abc_beta=0.95,
    xyz_gamma1=0.10,
    xyz_gamma2=0.25,
    transition_width=0.05,
)

print(f"Created {len(test_skus)} SKUs for the probabilistic classification example.")
print(f"The example includes {len(boundary_skus)} deliberately selected SKUs.")
print(
    "ABC cumulative-value cutoffs: "
    f"{prob_thresholds.abc_alpha:.0%} and {prob_thresholds.abc_beta:.0%}."
)
print(
    "XYZ CV cutoffs: "
    f"{prob_thresholds.xyz_gamma1:.0%} and {prob_thresholds.xyz_gamma2:.0%}."
)

## Run the classification and inspect the results

Each SKU receives marginal ABC and XYZ probability vectors, a nine-category joint distribution, one independently sampled label, a confidence index, and a boundary flag. The sampled label is one realization of the heuristic; the probability vectors retain information that the single label cannot show. The checks in the classification function verify that each probability vector sums to one.

In [ ]:
probabilistic_results = probabilistic_abc_xyz_classification(
    test_skus, prob_thresholds
)

key_sku_ids = {
    "PREMIUM_LAPTOP",
    "OFFICE_CHAIR",
    "SEASONAL_DECOR",
}
key_sku_ids.update(sku.sku_id for sku in boundary_skus)

print("DETAILED RESULTS FOR SELECTED SKUs")
print("=" * 72)
for result in probabilistic_results:
    if result.sku.sku_id not in key_sku_ids:
        continue

    sku = result.sku
    print(f"\n{sku.sku_id}:")
    print(f"  Annual value: ${sku.annual_value:,.2f}")
    print(f"  Demand coefficient of variation: {sku.cv:.3f}")
    print(f"  ABC probabilities: {result.abc_probabilities}")
    print(f"  XYZ probabilities: {result.xyz_probabilities}")
    print("  Three largest combined probabilities:")
    for category, probability in sorted(
        result.combined_probabilities.items(),
        key=lambda item: item[1],
        reverse=True,
    )[:3]:
        print(f"    {category}: {probability:.3f} ({probability:.1%})")
    print(f"  Sampled classification: {result.final_classification}")
    print(f"  Confidence index: {result.confidence_score:.3f}")
    print(f"  Boundary case: {'Yes' if result.is_boundary_case else 'No'}")

abc_counts = {category: 0 for category in ["A", "B", "C"]}
xyz_counts = {category: 0 for category in ["X", "Y", "Z"]}
combined_counts = {}
boundary_count = 0
high_confidence_count = 0

for result in probabilistic_results:
    abc_counts[result.final_classification[0]] += 1
    xyz_counts[result.final_classification[1]] += 1
    combined_counts[result.final_classification] = (
        combined_counts.get(result.final_classification, 0) + 1
    )
    boundary_count += int(result.is_boundary_case)
    high_confidence_count += int(result.confidence_score >= 0.8)

total_skus_classified = len(probabilistic_results)
avg_confidence = float(
    np.mean([result.confidence_score for result in probabilistic_results])
)

print("\nSAMPLED CLASSIFICATION SUMMARY")
print("=" * 72)
print("ABC distribution:")
for category in ["A", "B", "C"]:
    count = abc_counts[category]
    print(f"  {category}: {count} SKUs ({count / total_skus_classified:.1%})")

print("XYZ distribution:")
for category in ["X", "Y", "Z"]:
    count = xyz_counts[category]
    print(f"  {category}: {count} SKUs ({count / total_skus_classified:.1%})")

print("Summary measures:")
print(f"  Mean confidence index: {avg_confidence:.3f}")
print(
    f"  Confidence index at least 0.8: {high_confidence_count}/"
    f"{total_skus_classified} ({high_confidence_count / total_skus_classified:.1%})"
)
print(
    f"  Boundary cases: {boundary_count}/{total_skus_classified} "
    f"({boundary_count / total_skus_classified:.1%})"
)
print("Most frequent sampled combined categories:")
for category, count in sorted(
    combined_counts.items(), key=lambda item: item[1], reverse=True
)[:5]:
    print(f"  {category}: {count} SKUs ({count / total_skus_classified:.1%})")

## Visualize transitions, uncertainty, and repeatability

The first panel shows the ABC transition functions for illustrative value thresholds, so it explains the shape of the heuristic rather than displaying thresholds estimated from the generated portfolio. The remaining panels summarize the current classification run. The stability panel repeats classification on a subset using different seeds and compares each sampled label with the first run. The red bar in the sample probability chart identifies the sampled class; it does not mean that this class has the highest probability.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle(
    "Probabilistic ABC/XYZ Classification Analysis",
    fontsize=16,
    fontweight="bold",
)

# Illustrative thresholds make the Gaussian transition shapes easy to inspect.
ax1 = axes[0, 0]
values = np.linspace(0, 150000, 1000)
a_threshold_plot = 95000
b_threshold_plot = 18000
width_plot = 5000
p_a_values = [
    gaussian_transition_function(value, a_threshold_plot, width_plot, "above")
    for value in values
]
p_c_values = [
    gaussian_transition_function(value, b_threshold_plot, width_plot, "below")
    for value in values
]
p_b_values = [
    max(0.0, 1.0 - p_a - p_c)
    for p_a, p_c in zip(p_a_values, p_c_values)
]
ax1.plot(values, p_a_values, color="red", linewidth=2, label="A-class weight")
ax1.plot(values, p_b_values, color="green", linewidth=2, label="B-class weight")
ax1.plot(values, p_c_values, color="blue", linewidth=2, label="C-class weight")
ax1.axvline(
    a_threshold_plot, color="red", linestyle="--", alpha=0.6,
    label="Illustrative A threshold",
)
ax1.axvline(
    b_threshold_plot, color="blue", linestyle="--", alpha=0.6,
    label="Illustrative B threshold",
)
ax1.set_xlabel("Annual value (dollars)")
ax1.set_ylabel("Unnormalized class weight")
ax1.set_title("Gaussian Transition Weights for ABC")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(-0.05, 1.05)

ax2 = axes[0, 1]
confidences = [result.confidence_score for result in probabilistic_results]
ax2.hist(confidences, bins=20, alpha=0.7, color="skyblue", edgecolor="black")
ax2.axvline(
    0.8, color="red", linestyle="--", alpha=0.7,
    label="Reference index value: 0.8",
)
ax2.set_xlabel("Confidence index")
ax2.set_ylabel("Number of SKUs")
ax2.set_title("Confidence Index Distribution")
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = axes[0, 2]
boundary_data = [
    result.confidence_score
    for result in probabilistic_results
    if result.is_boundary_case
]
non_boundary_data = [
    result.confidence_score
    for result in probabilistic_results
    if not result.is_boundary_case
]
boxplot_data = []
boxplot_labels = []
if non_boundary_data:
    boxplot_data.append(non_boundary_data)
    boxplot_labels.append("Non-boundary")
if boundary_data:
    boxplot_data.append(boundary_data)
    boxplot_labels.append("Boundary")
if boxplot_data:
    ax3.boxplot(boxplot_data, labels=boxplot_labels)
ax3.set_ylabel("Confidence index")
ax3.set_title("Confidence by Boundary Flag")
ax3.grid(True, alpha=0.3)

ax4 = axes[1, 0]
matrix_data = np.zeros((3, 3), dtype=int)
abc_index = {"A": 0, "B": 1, "C": 2}
xyz_index = {"X": 0, "Y": 1, "Z": 2}
for result in probabilistic_results:
    matrix_data[
        abc_index[result.final_classification[0]],
        xyz_index[result.final_classification[1]],
    ] += 1

image = ax4.imshow(matrix_data, cmap="Blues", alpha=0.8)
fig.colorbar(image, ax=ax4, label="Number of SKUs")
ax4.set_xticks([0, 1, 2])
ax4.set_yticks([0, 1, 2])
ax4.set_xticklabels(["X (stable)", "Y (moderate)", "Z (volatile)"])
ax4.set_yticklabels(["A (high value)", "B (medium value)", "C (low value)"])
ax4.set_xlabel("XYZ class")
ax4.set_ylabel("ABC class")
ax4.set_title("Counts of Sampled ABC/XYZ Classes")
for row in range(3):
    for column in range(3):
        ax4.text(
            column,
            row,
            str(matrix_data[row, column]),
            ha="center",
            va="center",
            color="black",
            fontweight="bold",
        )

ax5 = axes[1, 1]
boundary_example = next(
    (
        result
        for result in probabilistic_results
        if result.is_boundary_case and result.sku.sku_id == "BUDGET_TABLET"
    ),
    None,
)
if boundary_example is None:
    boundary_example = next(
        (result for result in probabilistic_results if result.is_boundary_case),
        None,
    )
if boundary_example is not None:
    categories = list(boundary_example.combined_probabilities)
    probabilities = [
        boundary_example.combined_probabilities[category]
        for category in categories
    ]
    colors = [
        "red" if category == boundary_example.final_classification else "lightblue"
        for category in categories
    ]
    bars = ax5.bar(categories, probabilities, color=colors, alpha=0.8)
    ax5.set_ylabel("Combined probability")
    ax5.set_xlabel("ABC/XYZ category")
    ax5.set_title(
        f"Joint Distribution: {boundary_example.sku.sku_id}"
    )
    ax5.tick_params(axis="x", rotation=45)
    ax5.set_ylim(0, max(probabilities) * 1.2 if max(probabilities) > 0 else 1)
    for bar, probability in zip(bars, probabilities):
        ax5.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{probability:.2f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )
else:
    ax5.text(0.5, 0.5, "No boundary case in this run", ha="center", va="center")
    ax5.set_axis_off()

ax6 = axes[1, 2]
stability_results = []
n_runs = 10
stability_skus = test_skus[:10]
original_random_state = np.random.get_state()
reference_classifications = None

try:
    for run in range(n_runs):
        np.random.seed(run)
        run_results = probabilistic_abc_xyz_classification(
            stability_skus, prob_thresholds
        )
        current_classifications = {
            result.sku.sku_id: result.final_classification
            for result in run_results
        }

        if run == 0:
            reference_classifications = current_classifications
        else:
            matches = sum(
                current_classifications[sku_id] == reference_classifications[sku_id]
                for sku_id in reference_classifications
            )
            stability_results.append(matches / len(reference_classifications))
finally:
    # The diagnostic repeats use their own seeds and leave the main random stream
    # unchanged for any subsequent notebook calculations.
    np.random.set_state(original_random_state)

if stability_results:
    ax6.plot(
        range(2, n_runs + 1),
        stability_results,
        "bo-",
        linewidth=2,
        markersize=6,
        label="Share matching the first run",
    )
    ax6.axhline(
        0.85, color="red", linestyle="--", alpha=0.7,
        label="Illustrative 0.85 reference",
    )
    ax6.set_xlabel("Run number")
    ax6.set_ylabel("Fraction of matching labels")
    ax6.set_title("Repeatability for the First 10 SKUs")
    ax6.legend(fontsize=8)
    ax6.grid(True, alpha=0.3)
    ax6.set_ylim(0, 1)

plt.tight_layout()
plt.show()
print("Classification visualizations completed.")

## Deterministic reference comparison

This section uses the same cumulative-value cutoffs and XYZ CV boundaries to create a deterministic reference label for each SKU. The reference is used only to make the effect of probability-weighted sampling visible; it is not a replacement for the notebook's probabilistic heuristic. A mismatch between the sampled label and the reference is an expected possibility near transitions, not by itself evidence that either label is incorrect. The reported stability fraction compares repeated random draws with one reference draw and should be interpreted as a descriptive diagnostic.

In [ ]:
def deterministic_classification(
    skus: List[SKU],
    thresholds: ProbabilisticThresholds,
) -> List[str]:
    """Return threshold-based reference labels for comparison with sampled labels."""
    if not skus:
        raise ValueError("At least one SKU is required for classification.")

    total_value = sum(sku.annual_value for sku in skus)
    if total_value <= 0 or not np.isfinite(total_value):
        raise ValueError("Total annual value must be finite and positive.")

    sorted_skus = sorted(skus, key=lambda sku: sku.annual_value, reverse=True)
    cumulative_values = []
    running_total = 0.0
    for sku in sorted_skus:
        running_total += sku.annual_value
        cumulative_values.append(running_total / total_value)

    a_threshold_idx = next(
        i for i, share in enumerate(cumulative_values)
        if share >= thresholds.abc_alpha
    )
    b_threshold_idx = next(
        i for i, share in enumerate(cumulative_values)
        if share >= thresholds.abc_beta
    )
    a_threshold = sorted_skus[a_threshold_idx].annual_value
    b_threshold = sorted_skus[b_threshold_idx].annual_value

    classifications = []
    for sku in skus:
        if sku.annual_value >= a_threshold:
            abc_class = "A"
        elif sku.annual_value >= b_threshold:
            abc_class = "B"
        else:
            abc_class = "C"

        if sku.cv <= thresholds.xyz_gamma1:
            xyz_class = "X"
        elif sku.cv <= thresholds.xyz_gamma2:
            xyz_class = "Y"
        else:
            xyz_class = "Z"
        classifications.append(abc_class + xyz_class)

    return classifications

reference_labels = {
    sku.sku_id: label
    for sku, label in zip(
        test_skus,
        deterministic_classification(test_skus, prob_thresholds),
    )
}

boundary_comparisons = []
for result in probabilistic_results:
    if not result.is_boundary_case:
        continue
    reference_label = reference_labels[result.sku.sku_id]
    if reference_label != result.final_classification:
        boundary_comparisons.append(
            {
                "sku": result.sku.sku_id,
                "reference": reference_label,
                "sampled": result.final_classification,
                "confidence": result.confidence_score,
            }
        )

print("BOUNDARY-CASE REFERENCE COMPARISON")
print("=" * 72)
if boundary_comparisons:
    print(
        f"Boundary cases with a sampled label different from the reference: "
        f"{len(boundary_comparisons)}."
    )
    for comparison in boundary_comparisons[:10]:
        print(
            f"  {comparison['sku']}: reference={comparison['reference']}, "
            f"sampled={comparison['sampled']}, "
            f"confidence index={comparison['confidence']:.3f}"
        )
else:
    print("No sampled-label differences were found among flagged boundary cases.")

print("\nInterpretation:")
print(
    "The reference uses hard thresholds, whereas the heuristic samples from "
    "transition-based weights. The comparison describes their labels; it does "
    "not measure classification accuracy."
)
if stability_results:
    print(
        "Mean repeatability fraction for the diagnostic runs: "
        f"{np.mean(stability_results):.3f}."
    )
    print(
        "This fraction compares each diagnostic run with its first run for "
        "the selected subset; it is not a performance guarantee."
    )

## Summary and conclusions

This notebook implements a probabilistic ABC/XYZ classification heuristic. Cumulative portfolio value determines the ABC value cutoffs; CV thresholds determine the XYZ boundaries. Gaussian CDF transitions assign class weights around those boundaries, and independent weighted draws produce the reported ABC and XYZ labels. The products of the marginal weights form the nine combined category probabilities under the stated independence assumption.

The probability vectors preserve ambiguity that a single sampled label cannot show. The confidence index summarizes concentration in those vectors, while the boundary flag identifies cases whose strongest marginal class weight is below the specified cutoff. Neither measure is calibrated against observed classification outcomes. Repeated-run consistency describes the behavior of randomized labels for the selected data and seeds; it does not establish general stability.

Use the outputs as decision-support summaries within this model's scope. The generated example demonstrates the method, not a validated inventory policy. Applying the classifications operationally requires appropriate business thresholds, reliable value and demand data, and domain review.